In [0]:

# =============================================================================
# NOTEBOOK: load_team_mappings_to_silver
# =============================================================================
#
# PURPOSE
# -------
# Incrementally load HUMAN-APPROVED canonical teams and source mappings
# into the existing Silver Delta reference tables.
#
# TARGET TABLES
# -------------
# football_data_platform.silver.canonical_teams
# football_data_platform.silver.team_mappings
#
# WHEN TO RUN
# -----------
# Run only when extract_team_names reports unmapped source identities
# and the proposed additions have been reviewed.
#
# REQUIRED INPUT FILES
# --------------------
# 1. canonical_teams_additions.csv
#      team_name
#
#    Contains only genuinely new canonical clubs.
#    A header-only file is valid when no new canonical club is needed.
#
# 2. team_mappings_additions.csv
#      source_system,source_team_id,source_team_name,canonical_team_name
#
#    Contains one reviewed assignment per new source identity.
#
# FILE LOCATION
# -------------
# /Volumes/football_data_platform/raw/source_files/
# team_standardization_review/
#
# IMPORTANT RULES
# ---------------
# - Do not assign team_key manually.
# - Never overwrite an existing canonical team_key.
# - Never automatically reassign an existing source identity.
# - Do not automatically fuzzy-match new teams.
# - Run only one reference-data loader at a time.
# - Repeated runs of the same approved input must not duplicate records.
#
# AFTER SUCCESS
# -------------
# 1. Rerun extract_team_names.
# 2. Confirm zero unmapped source identities.
# 3. Run the affected Silver transformations.
# 4. Archive the processed review files.
#
# NOTE ON TRANSACTIONS
# --------------------
# Canonical teams and source mappings are two separate Delta tables.
# Their MERGE operations are not one atomic transaction.
# This notebook is designed so a rerun can recover missing inserts.
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable


# =============================================================================
# 1. Configuration
# =============================================================================

CATALOG = "football_data_platform"
SILVER_SCHEMA = f"{CATALOG}.silver"

CANONICAL_TABLE = f"{SILVER_SCHEMA}.canonical_teams"
MAPPINGS_TABLE = f"{SILVER_SCHEMA}.team_mappings"

REVIEW_PATH = (
    "/Volumes/football_data_platform/raw/source_files/"
    "team_standardization_review"
)

CANONICAL_PATH = (
    f"{REVIEW_PATH}/canonical_teams_additions.csv"
)

MAPPINGS_PATH = (
    f"{REVIEW_PATH}/team_mappings_additions.csv"
)

API_SOURCE = "football-data.org"

SUPPORTED_SOURCES = [
    "football-data.org",
    "fbref",
    "football-data.co.uk",
]

CANONICAL_INPUT_COLUMNS = [
    "team_name",
]

MAPPING_INPUT_COLUMNS = [
    "source_system",
    "source_team_id",
    "source_team_name",
    "canonical_team_name",
]


# =============================================================================
# 2. Helpers
# =============================================================================

def fail_if_rows_exist(df, message):
    """
    Raise an exception if a DataFrame contains at least one row.
    """

    if df.limit(1).count() > 0:
        raise ValueError(message)


def require_columns(df, required_columns, label):
    """
    Validate that all required columns are present.
    """

    missing = [
        column_name
        for column_name in required_columns
        if column_name not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{label} is missing required columns: {missing}"
        )


def add_source_identity(df):
    """
    Construct the same matching identity used by Silver.

    football-data.org:
        source_system + source_team_id

    fbref / football-data.co.uk:
        source_system + source_team_name
    """

    return df.withColumn(
        "_identity_key",
        F.when(
            F.col("source_system") == API_SOURCE,
            F.col("source_team_id"),
        ).otherwise(
            F.col("source_team_name")
        ),
    )


def has_rows(df):
    return df.limit(1).count() > 0


# =============================================================================
# 3. Start
# =============================================================================

print("=" * 70)
print("TEAM STANDARDIZATION — APPROVED INCREMENTAL LOAD")
print("=" * 70)

print("\nThis notebook updates persistent Silver reference tables.")
print("Only reviewed and approved additions should be loaded.")

print("\nSTEP 1: Checking existing Delta tables.")

for table in [CANONICAL_TABLE, MAPPINGS_TABLE]:

    if not spark.catalog.tableExists(table):
        raise ValueError(
            f"Required table does not exist: {table}. "
            "Complete the initial reference-table setup first."
        )

print("Existing reference tables found.")


# =============================================================================
# 4. Read reviewed CSV files
# =============================================================================
#
# Both files must exist.
#
# A header-only canonical additions file means that every source mapping
# points to a club which already has a canonical key.
# =============================================================================

print("\nSTEP 2: Reading approved CSV additions.")

canonical_input_df = (
    spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .csv(CANONICAL_PATH)
)

mapping_input_df = (
    spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .csv(MAPPINGS_PATH)
)

require_columns(
    canonical_input_df,
    CANONICAL_INPUT_COLUMNS,
    "Canonical additions CSV",
)

require_columns(
    mapping_input_df,
    MAPPING_INPUT_COLUMNS,
    "Mapping additions CSV",
)

# Keep the input contract deliberately narrow.
# Unexpected columns may indicate the wrong file was uploaded.

if set(canonical_input_df.columns) != set(CANONICAL_INPUT_COLUMNS):
    raise ValueError(
        "canonical_teams_additions.csv must contain only: team_name"
    )

if set(mapping_input_df.columns) != set(MAPPING_INPUT_COLUMNS):
    raise ValueError(
        "team_mappings_additions.csv must contain exactly: "
        + ", ".join(MAPPING_INPUT_COLUMNS)
    )


# =============================================================================
# 5. Clean input values
# =============================================================================

canonical_input_df = (
    canonical_input_df
    .select(
        F.trim(
            F.col("team_name")
        ).alias("team_name")
    )
)

mapping_input_df = (
    mapping_input_df
    .select(
        F.trim(
            F.col("source_system")
        ).alias("source_system"),

        F.trim(
            F.col("source_team_id").cast("string")
        ).alias("source_team_id"),

        F.trim(
            F.col("source_team_name")
        ).alias("source_team_name"),

        F.trim(
            F.col("canonical_team_name")
        ).alias("canonical_team_name"),
    )
)

# Convert empty optional source IDs to NULL.
mapping_input_df = mapping_input_df.withColumn(
    "source_team_id",
    F.when(
        F.col("source_team_id") == "",
        F.lit(None).cast("string"),
    ).otherwise(
        F.col("source_team_id")
    ),
)

print("CSV files loaded.")


# =============================================================================
# 6. Validate approved input files
# =============================================================================

print("\nSTEP 3: Validating approved input.")

# The canonical additions file may be empty (header-only).
# However, it must not contain blank rows disguised as new clubs.

fail_if_rows_exist(
    canonical_input_df.filter(
        F.col("team_name").isNull()
        | (F.col("team_name") == "")
    ),
    "Canonical additions contain blank team names.",
)

# Mapping additions must contain at least one reviewed source identity.
if not has_rows(mapping_input_df):

    print("No approved mappings were supplied.")
    print("No reference data was changed.")
    print("NEXT: Run extract_team_names to check current coverage.")

    raise ValueError(
        "team_mappings_additions.csv contains no mapping rows."
    )

# Required mapping attributes must be populated.
for column_name in [
    "source_system",
    "source_team_name",
    "canonical_team_name",
]:

    fail_if_rows_exist(
        mapping_input_df.filter(
            F.col(column_name).isNull()
            | (F.col(column_name) == "")
        ),
        f"Missing required value: {column_name}",
    )

# Reject unexpected source-system values.
fail_if_rows_exist(
    mapping_input_df.filter(
        ~F.col("source_system").isin(SUPPORTED_SOURCES)
    ),
    "Unsupported source_system in mapping additions.",
)

# API mappings require stable provider team IDs.
fail_if_rows_exist(
    mapping_input_df.filter(
        (F.col("source_system") == API_SOURCE)
        & F.col("source_team_id").isNull()
    ),
    "football-data.org mappings require source_team_id.",
)

# The current football-data.org team IDs are numeric.
fail_if_rows_exist(
    mapping_input_df.filter(
        (F.col("source_system") == API_SOURCE)
        & ~F.col("source_team_id").rlike(r"^[0-9]+$")
    ),
    "Invalid football-data.org source_team_id.",
)

# Name-based sources should not invent source IDs.
fail_if_rows_exist(
    mapping_input_df.filter(
        (F.col("source_system") != API_SOURCE)
        & F.col("source_team_id").isNotNull()
    ),
    "FBref and football-data.co.uk must use blank source_team_id.",
)

# New canonical names must not repeat inside the additions file.
fail_if_rows_exist(
    canonical_input_df
    .groupBy("team_name")
    .count()
    .filter(F.col("count") > 1),
    "Duplicate canonical names in additions file.",
)

# Every incoming source identity must be unique.
mapping_input_df = add_source_identity(mapping_input_df)

fail_if_rows_exist(
    mapping_input_df
    .groupBy(
        "source_system",
        "_identity_key",
    )
    .count()
    .filter(F.col("count") > 1),
    "Duplicate source identities in mapping additions.",
)

print("Input validation passed.")


# =============================================================================
# 7. Read and validate existing canonical teams
# =============================================================================

print("\nSTEP 4: Validating existing reference data.")

canonical_current_df = (
    spark.table(CANONICAL_TABLE)
    .select(
        F.col("team_key").cast("long").alias("team_key"),
        F.col("team_name").alias("team_name"),
    )
)

fail_if_rows_exist(
    canonical_current_df.filter(
        F.col("team_key").isNull()
        | F.col("team_name").isNull()
        | (F.trim(F.col("team_name")) == "")
    ),
    "Existing canonical_teams contains incomplete records.",
)

fail_if_rows_exist(
    canonical_current_df
    .groupBy("team_key")
    .count()
    .filter(F.col("count") > 1),
    "Existing canonical_teams contains duplicate team_key values.",
)

fail_if_rows_exist(
    canonical_current_df
    .groupBy("team_name")
    .count()
    .filter(F.col("count") > 1),
    "Existing canonical_teams contains duplicate names.",
)


# =============================================================================
# 8. Read and validate existing mappings
# =============================================================================

existing_mappings_df = (
    spark.table(MAPPINGS_TABLE)
    .select(
        "source_system",

        F.col("source_team_id")
        .cast("string")
        .alias("source_team_id"),

        "source_team_name",

        F.col("team_key")
        .cast("long")
        .alias("team_key"),
    )
)

existing_mappings_df = add_source_identity(
    existing_mappings_df
)

fail_if_rows_exist(
    existing_mappings_df.filter(
        F.col("source_system").isNull()
        | F.col("_identity_key").isNull()
        | (F.col("_identity_key") == "")
        | F.col("team_key").isNull()
    ),
    "Existing team_mappings contains incomplete identities.",
)

fail_if_rows_exist(
    existing_mappings_df
    .groupBy("source_system", "_identity_key")
    .count()
    .filter(F.col("count") > 1),
    "Existing team_mappings contains duplicate identities.",
)

# Validate that existing mapping keys belong to existing canonical clubs.
fail_if_rows_exist(
    existing_mappings_df
    .select("team_key")
    .distinct()
    .join(
        canonical_current_df.select("team_key"),
        on="team_key",
        how="left_anti",
    ),
    "Existing mappings reference nonexistent canonical teams.",
)


# =============================================================================
# 9. Validate mapping assignments
# =============================================================================
#
# All canonical names referenced in the incoming mapping file must be:
#   A. Already present in Silver, or
#   B. Explicitly supplied in canonical_teams_additions.csv.
#
# Existing source identities cannot be reassigned here.
# =============================================================================

available_names_df = (
    canonical_current_df
    .select("team_name")
    .unionByName(canonical_input_df)
    .distinct()
)

unknown_canonical_df = (
    mapping_input_df
    .select(
        F.col("canonical_team_name").alias("team_name")
    )
    .distinct()
    .join(
        available_names_df,
        on="team_name",
        how="left_anti",
    )
)

if has_rows(unknown_canonical_df):

    display(unknown_canonical_df)

    raise ValueError(
        "Some mappings reference unknown canonical names. "
        "Add approved new clubs to canonical_teams_additions.csv "
        "or correct their names to match existing canonical teams."
    )

# Resolve existing source identities to their current canonical names.
existing_named_df = (
    existing_mappings_df
    .join(
        canonical_current_df.select(
            "team_key",
            F.col("team_name").alias(
                "existing_canonical_name"
            ),
        ),
        on="team_key",
        how="left",
    )
)

# Reject changes to an identity which is already assigned.
conflicts_df = (
    mapping_input_df.alias("incoming")
    .join(
        existing_named_df.alias("existing"),
        on=["source_system", "_identity_key"],
        how="inner",
    )
    .filter(
        F.col("incoming.canonical_team_name")
        != F.col("existing.existing_canonical_name")
    )
)

if has_rows(conflicts_df):

    display(
        conflicts_df.select(
            "source_system",
            "_identity_key",
            "canonical_team_name",
            "existing_canonical_name",
        )
    )

    raise ValueError(
        "A source identity already maps to a different canonical team. "
        "Existing mappings cannot be reassigned by this loader."
    )

# Do not silently rename existing canonical teams.
# A canonical name already present in Silver is reused.
# A new name receives a new key only if explicitly approved.


# =============================================================================
# 10. Prepare genuinely new canonical teams
# =============================================================================

print("\nSTEP 5: Preparing canonical team additions.")

new_canonical_df = (
    canonical_input_df
    .join(
        canonical_current_df.select("team_name"),
        on="team_name",
        how="left_anti",
    )
)

# Allocate new team keys after the highest existing key.
#
# Single-writer assumption:
# Do not run concurrent instances of this loader.
#
# Ordering by team_name makes assignments deterministic within a batch.

max_team_key = (
    canonical_current_df
    .agg(
        F.coalesce(
            F.max("team_key"),
            F.lit(0),
        ).alias("max_team_key")
    )
    .first()["max_team_key"]
)

key_window = Window.orderBy("team_name")

new_canonical_df = (
    new_canonical_df
    .withColumn(
        "team_key",
        F.row_number().over(key_window)
        + F.lit(max_team_key),
    )
    .select(
        F.col("team_key").cast("long").alias("team_key"),
        "team_name",
    )
)

# Complete lookup covering old and newly approved clubs.
all_canonical_df = (
    canonical_current_df
    .unionByName(new_canonical_df)
)


# =============================================================================
# 11. Prepare new source mappings
# =============================================================================

print("\nSTEP 6: Preparing source mapping additions.")

resolved_mappings_df = (
    mapping_input_df
    .join(
        all_canonical_df.select(
            F.col("team_name").alias("canonical_team_name"),
            "team_key",
        ),
        on="canonical_team_name",
        how="left",
    )
)

fail_if_rows_exist(
    resolved_mappings_df.filter(
        F.col("team_key").isNull()
    ),
    "An approved mapping could not resolve to a canonical key.",
)

# Exclude source identities which are already present.
#
# Rerunning a previously applied mapping file must produce no new
# mapping records, rather than appending duplicates.

new_mappings_df = (
    resolved_mappings_df
    .join(
        existing_mappings_df.select(
            "source_system",
            "_identity_key",
        ),
        on=["source_system", "_identity_key"],
        how="left_anti",
    )
)

# Preserve the metadata columns used in the existing team_mappings table.
# Fields such as league_codes and source_tables remain optional metadata.
#
# The mapping loader does not invent historical source coverage.

new_mappings_df = (
    new_mappings_df
    .withColumn(
        "league_codes",
        F.lit(None).cast("string"),
    )
    .withColumn(
        "source_tables",
        F.lit(None).cast("string"),
    )
    .withColumn(
        "match_method",
        F.lit("manual_review"),
    )
    .withColumn(
        "review_status",
        F.lit("approved"),
    )
)

canonical_to_insert = new_canonical_df.count()
mappings_to_insert = new_mappings_df.count()

print(f"New canonical clubs: {canonical_to_insert}")
print(f"New source mappings: {mappings_to_insert}")


# =============================================================================
# 12. Check existing target schemas before writing
# =============================================================================
#
# These checks prevent an unexpected Delta target schema from causing
# the second MERGE to fail after the first table has already changed.
# =============================================================================

canonical_target_columns = spark.table(CANONICAL_TABLE).columns
mapping_target_columns = spark.table(MAPPINGS_TABLE).columns

if set(canonical_target_columns) != {"team_key", "team_name"}:
    raise ValueError(
        "Unexpected canonical_teams schema. "
        f"Actual columns: {canonical_target_columns}. "
        "Adapt the loader explicitly before writing."
    )

supported_mapping_columns = {
    "source_system",
    "source_team_id",
    "source_team_name",
    "team_key",
    "canonical_team_name",
    "league_codes",
    "source_tables",
    "match_method",
    "review_status",
}

required_target_mapping_columns = {
    "source_system",
    "source_team_id",
    "source_team_name",
    "team_key",
}

if not required_target_mapping_columns.issubset(
    set(mapping_target_columns)
):
    raise ValueError(
        "Existing team_mappings table lacks required columns."
    )

unexpected_target_columns = (
    set(mapping_target_columns) - supported_mapping_columns
)

if unexpected_target_columns:
    raise ValueError(
        "Unexpected team_mappings columns: "
        f"{unexpected_target_columns}. "
        "Define their values before modifying the table."
    )

# Ensure canonical_team_name, when stored in the mapping table,
# matches the approved canonical name.

# Select only the columns present in the existing target.
new_mappings_for_write_df = (
    new_mappings_df
    .select(*mapping_target_columns)
)

print("\nAll pre-write validations completed.")


# =============================================================================
# 13. Apply insert-only Delta MERGEs
# =============================================================================
#
# IMPORTANT:
# All mapping and canonical validations happen before this section.
#
# No existing assignments are updated.
# Repeated approved inputs should become no-ops.
#
# Two separate tables cannot be updated in one Delta transaction.
# If the second MERGE fails, correct the issue and rerun this notebook.
# =============================================================================

print("\nSTEP 7: Applying Delta MERGEs.")

if canonical_to_insert > 0:

    canonical_delta = DeltaTable.forName(
        spark,
        CANONICAL_TABLE,
    )

    (
        canonical_delta.alias("target")
        .merge(
            new_canonical_df.alias("source"),
            """
            target.team_key = source.team_key
            OR target.team_name = source.team_name
            """,
        )
        .whenNotMatchedInsert(
            values={
                "team_key": "source.team_key",
                "team_name": "source.team_name",
            }
        )
        .execute()
    )

    print(
        f"Canonical MERGE completed: "
        f"{canonical_to_insert} proposed insertions."
    )

else:

    print("No new canonical clubs to insert.")


# Matching condition must follow the source's identity strategy.
#
# API:
#   source_system + source_team_id
#
# FBref / football-data.co.uk:
#   source_system + source_team_name

mapping_merge_condition = """
target.source_system = source.source_system
AND
(
    (
        target.source_system = 'football-data.org'
        AND target.source_team_id = source.source_team_id
    )
    OR
    (
        target.source_system <> 'football-data.org'
        AND target.source_team_name = source.source_team_name
    )
)
"""

if mappings_to_insert > 0:

    mappings_delta = DeltaTable.forName(
        spark,
        MAPPINGS_TABLE,
    )

    insert_values = {
        column_name: f"source.{column_name}"
        for column_name in mapping_target_columns
    }

    (
        mappings_delta.alias("target")
        .merge(
            new_mappings_for_write_df.alias("source"),
            mapping_merge_condition,
        )
        .whenNotMatchedInsert(
            values=insert_values
        )
        .execute()
    )

    print(
        f"Mappings MERGE completed: "
        f"{mappings_to_insert} proposed insertions."
    )

else:

    print("No new source mappings to insert.")


# =============================================================================
# 14. Final verification
# =============================================================================

print("\nSTEP 8: Verifying reference-data integrity.")

final_canonical_df = spark.table(CANONICAL_TABLE)
final_mappings_df = spark.table(MAPPINGS_TABLE)

fail_if_rows_exist(
    final_mappings_df
    .select("team_key")
    .distinct()
    .join(
        final_canonical_df.select("team_key"),
        on="team_key",
        how="left_anti",
    ),
    "Final reference integrity check failed.",
)

# All approved source identities must now be present and point to the
# expected canonical team keys.

final_identity_df = add_source_identity(
    final_mappings_df.select(
        "source_system",
        "source_team_id",
        "source_team_name",
        "team_key",
    )
)

missing_approved_df = (
    resolved_mappings_df
    .select(
        "source_system",
        "_identity_key",
        "team_key",
    )
    .join(
        final_identity_df.select(
            "source_system",
            "_identity_key",
            "team_key",
        ),
        on=[
            "source_system",
            "_identity_key",
            "team_key",
        ],
        how="left_anti",
    )
)

fail_if_rows_exist(
    missing_approved_df,
    "Not all approved team mappings were persisted.",
)

print("Final reference-data validation passed.")


# =============================================================================
# 15. Instructions for the next operator step
# =============================================================================

print("\n" + "=" * 70)
print("TEAM STANDARDIZATION — LOAD COMPLETED")
print("=" * 70)

print(f"Canonical teams: {final_canonical_df.count()}")
print(f"Team mappings: {final_mappings_df.count()}")

if canonical_to_insert == 0 and mappings_to_insert == 0:

    print("\nNo new records were required.")
    print("The approved input was already represented in Silver.")
    print("This is the expected result of an idempotent rerun.")

else:

    print("\nApproved reference-data additions have been processed.")

print("\nNEXT STEPS:")
print("1. Rerun extract_team_names against the current Bronze data.")
print("2. Confirm it reports zero unmapped source identities.")
print("3. If any remain, review those identities and repeat onboarding.")
print("4. When discovery is clean, run the Silver transformations.")
print("5. Archive the approved CSV files after successful verification.")
print("6. Commit reviewed reference-data changes through Git.")

print("\nIMPORTANT:")
print("- Existing canonical team keys were not reassigned.")
print("- Existing source mappings were not overwritten.")
print("- No fuzzy matching was performed.")
print("- Do not run multiple instances of this loader concurrently.")

print("\nIncremental team reference-data load completed.")
